<a href="https://colab.research.google.com/github/ja2faded/ds2002-fa26/blob/main/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [32]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-99', #changed based off TA suggestion; intentionally absent from vendors_json
   'lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one: item within an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [33]:
import pandas as pd

#record_path identifies nested list (will become rows); meta keeps original order and vendor id
lines = pd.json_normalize(orders_json, record_path='lines',
                  meta=[
                      'order', 'vendor_id'
                  ])


assert len(lines) == 4
assert lines['qty'].sum() == 7

lines
#matches expected 4 rows adn qty sum up to 7

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-99


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [34]:
#convert vendor records to df
vendors = pd.json_normalize(vendors_json)

#left join (keeps line item even if missing vendor)
#indicator= True creates _merge column
joined = lines.merge(vendors, on= 'vendor_id',
        how='left', indicator=True)


#find line items with no matching vendor
unmatched = joined[joined['_merge'].eq('left_only')]

print('Unmatched vendors: ', unmatched['vendor_id'].unique())
print('Unmatched qty: ', unmatched['qty'].sum())


unmatched
#V-99 is unmatched


Unmatched vendors:  ['V-99']
Unmatched qty:  3


,item,qty,order,vendor_id,name,zone,_merge
3,Hot Dog,3,3,V-99,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [35]:
#replace missing vendor name with "Unmatched: + vendor_id "
#so V-99's qty stays in the total/ its name still appears in the table
joined['vendor_label'] = joined['name'].fillna('Unmatched: ' + joined['vendor_id'])

#add the qtys for all line items for each vendor
vendor_totals = joined.groupby('vendor_label')['qty'].sum()

vendor_totals



,qty
vendor_label,
Cav Merch,1
Hoos Burgers,3
Unmatched: V-99,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [36]:
#group by zone and sum qtys for each
zone_totals = (joined.groupby('zone', as_index=False)['qty']
               .sum().sort_values('qty', ascending=False))

print(zone_totals)

#count units assigned to each zone
known_zone_units = zone_totals['qty'].sum()

#count all units
all_units = joined['qty'].sum()

missing_zone_units = all_units - known_zone_units

print('Zone total: ', known_zone_units)
print('Missing zone total: ', missing_zone_units)


  zone  qty
0    A    4
Zone total:  4
Missing zone total:  3


Zone A sold the most items (4 units). The known zone total is 4 since the unmatched vendor (V-99) has no zone and sold 3 units. This is indicated by the missing zone total being 3.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [37]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01',
     'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]


#find orders with a lines list
orders_with_lines = [
    orders for orders in ragged_json
    if 'lines' in orders
]


#flatten other orders to one row per line item
ragged_lines = pd.json_normalize(
    orders_with_lines, record_path='lines',meta=['order', 'vendor_id']
)


#can't use fillna(0) since that would indicate that a qty was reported, just that the value was 0
#using notna() tells you if an amount was reported or not
ragged_lines['qty_reported'] = ragged_lines['qty'].notna()

ragged_lines






,item,qty,order,vendor_id,qty_reported
0,Soda,2.0,4,V-01,True
1,Fries,NaN,6,V-01,False


### Q6 — Validate

In [38]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a. One row per line item is a useful shape here because each item is viewed as one observation, while preserving the order and vendor information. This makes the question: What is the total quantity of a specific item (also could do by vendor). If we had it so that every order was one row, the items within each order would be a list within the row, making it hard to get the totals across orders and vendors.

b. A missing quantity means that the quantity was not reported. A quantity of 0 means the quantity was reported and its number was 0. Conflating these (replacing a missing quantity with 0) would lead to errors when calculating averages and other statistics about each item/vendor downstream since it would be reporting 0 for something that is missing information.  